# PD04 · Limpieza e integración territorial

Continuación de PD01–PD03 basada en las clases del 9 y 10 de octubre de 2026. Tiempo orientativo: 270 minutos, con el entorno y los datos preparados.

Copia esta plantilla a `soluciones/` conservando su nombre. Escribe allí el código y las interpretaciones. Las celdas iniciales muestran las lecturas; las actividades quedan a tu cargo. Antes de entregar, reinicia el kernel y ejecuta todas las celdas en orden.

Utiliza directamente las API de pandas, GeoPandas y, en PD05, Polars. Mantén visibles las conversiones, máscaras, agrupaciones, uniones y exportaciones. No uses `talleres.py`, `pqrs_talleres.py`, comandos de shell ni funciones que resuelvan varias etapas sin mostrar su implementación. Un bucle de medición es válido si su cuerpo contiene las operaciones que estás midiendo. No instales paquetes desde el notebook.

[Índice](../ejercicios/README.md) · [Pandas](../docs/manual-pandas.md) · [Polars](../docs/manual-polars.md) · [GeoPandas](../docs/manual-geopandas.md)

## Ubicación de trabajo

Abre Jupyter desde la raíz del repositorio con el entorno del curso activo. Cambia `REPO` por la ruta real de tu copia. Las rutas son explícitas y funcionan tanto desde `ejercicios/` como desde `soluciones/`. Si falta un archivo, revisa su ubicación; no sustituyas silenciosamente la entrada por otro dataset.

## Pregunta y entradas

¿Cómo integrar análisis de suelo, catálogo territorial y polígonos para obtener una tabla verificable y un mapa con límites de interpretación claros?

| Fuente | Lectura directa | Uso |
|---|---|---|
| `agrosavia.csv` | `pd.read_csv` | Análisis, fechas y medidas con tipos y marcadores variados. |
| `divipola.csv` | `pd.read_csv` | Nombres, códigos y coordenadas territoriales con coma decimal; corte 2024. |
| `dane_municipios.zip` | `gpd.read_file` | Polígonos municipales MGN 2025 y código `mpio_cdpmp`. |
| `igac_capacidad.json` | `gpd.read_file` | Muestra GeoJSON de cinco polígonos de capacidad de uso. |

Primero integra las dos tablas con pandas. Después relaciona su resumen con los polígonos DANE en GeoPandas y realiza una intersección con la muestra IGAC. Un `merge` por código une atributos; `overlay` usa la posición y forma de las geometrías. Explica esa diferencia en tus resultados.

Las coordenadas de DIVIPOLA no son ubicaciones de muestras de suelo. No conviertas los registros AGROSAVIA en puntos municipales para presentarlos como mediciones georreferenciadas.

Prepara el ZIP siguiendo la [descarga manual DANE](../docs/entorno.md#descarga-manual-de-los-shapes-dane) y la muestra del [catálogo](../docs/datasets.md). Conserva juntos los componentes del Shapefile dentro del ZIP. Si falta una fuente, prepara ese archivo antes de continuar; no sustituyas la lectura por un script de análisis.

`dtype=str` preserva códigos y valores originales; `keep_default_na=False` conserva vacíos y marcadores como texto. Revisa las opciones de cada lectura.

In [1]:
import sys
print(sys.executable)

/mnt/c/Users/TUPTC/bigdata/big-data-postgraduate/.venv/bin/python


In [2]:
import os
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

REPO = "/mnt/c/Users/TUPTC/bigdata/big-data-postgraduate"
RAW = REPO + "/kit/data/raw"
OUT = REPO + "/kit/salidas/soluciones/PD04"
os.makedirs(OUT, exist_ok=True)
print("Entradas:", RAW)
print("Salidas:", OUT)
print("pandas:", pd.__version__, "GeoPandas:", gpd.__version__)

Entradas: /mnt/c/Users/TUPTC/bigdata/big-data-postgraduate/kit/data/raw
Salidas: /mnt/c/Users/TUPTC/bigdata/big-data-postgraduate/kit/salidas/soluciones/PD04
pandas: 2.2.3 GeoPandas: 1.2.0


## 1. Diagnóstico y contrato · 20 minutos

Muestra dimensiones, columnas y tipos de ambas tablas. Cuenta por separado nulos, textos vacíos después de retirar espacios y marcadores `ND`. Examina fecha, pH, nombres territoriales y coordenadas.

Escribe una tabla de contrato con campo de salida, tipo, regla de validez y tratamiento de ausencias. Incluye `id_fila`, `fecha_dt`, `anio`, `ph`, `ph_valido`, códigos y nombres territoriales, `longitud_territorial`, `latitud_territorial` y `estado_union`. Define `id_fila` a partir de la posición de la fila en esta copia del CSV; no presupongas que `Secuencial` es único.

**Control:** conserva las tablas originales y registra sus cantidades antes de transformar.

In [3]:
import numpy as np
import unicodedata

def norm_encabezado(texto):
    """Encabezado en minúsculas, sin tildes y con espacios/guiones bajos unificados (solo para ubicar columnas)."""
    t = unicodedata.normalize("NFKD", str(texto)).encode("ascii", "ignore").decode("ascii")
    return " ".join(t.lower().replace("_", " ").split())

def buscar_columna(df, candidatos, etiqueta, nombre_tabla):
    """Devuelve el nombre REAL de la columna cuyo encabezado normalizado coincide con algún candidato."""
    por_norm = {norm_encabezado(c): c for c in df.columns}
    for cand in candidatos:
        if norm_encabezado(cand) in por_norm:
            return por_norm[norm_encabezado(cand)]
    raise KeyError(f"No encuentro la columna de {etiqueta} en {nombre_tabla}. "
                   f"Columnas disponibles: {df.columns.tolist()}")

# --- Tabla de suelo (AGROSAVIA) ---
S_DEP = buscar_columna(suelo, ["Departamento", "Nombre Departamento", "Dpto"], "departamento", "suelo")
S_MUN = buscar_columna(suelo, ["Municipio", "Nombre Municipio", "Mpio"], "municipio", "suelo")
S_FECHA = buscar_columna(suelo, ["Fecha de Análisis", "Fecha Analisis", "Fecha"], "fecha de análisis", "suelo")
S_PH = buscar_columna(suelo, ["pH agua:suelo", "pH agua suelo", "pH"], "pH", "suelo")
S_SEC = buscar_columna(suelo, ["Secuencial", "Id", "Consecutivo"], "secuencial", "suelo")

# --- Tabla territorial (DIVIPOLA) ---
T_CDEP = buscar_columna(territorio, ["Código Departamento", "Cod Departamento", "Codigo Dpto", "Cod Dpto"], "código de departamento", "territorio")
T_CMUN = buscar_columna(territorio, ["Código Municipio", "Cod Municipio", "Codigo Mpio", "Cod Mpio"], "código de municipio", "territorio")
T_DEP = buscar_columna(territorio, ["Nombre Departamento", "Departamento", "Nom Dpto", "Dpto"], "nombre de departamento", "territorio")
T_MUN = buscar_columna(territorio, ["Nombre Municipio", "Municipio", "Nom Mpio", "Mpio"], "nombre de municipio", "territorio")
T_LON = buscar_columna(territorio, ["Longitud", "Lon", "Long"], "longitud", "territorio")
T_LAT = buscar_columna(territorio, ["Latitud", "Lat"], "latitud", "territorio")

print("Columnas usadas en suelo:      ", {"dep": S_DEP, "mun": S_MUN, "fecha": S_FECHA, "ph": S_PH, "sec": S_SEC})
print("Columnas usadas en territorio: ", {"cdep": T_CDEP, "cmun": T_CMUN, "dep": T_DEP, "mun": T_MUN, "lon": T_LON, "lat": T_LAT})
print("Columnas suelo:", suelo.columns.tolist())
print("Columnas territorio:", territorio.columns.tolist())

NameError: name 'suelo' is not defined

In [ ]:
# Cantidades de las tablas originales, registradas antes de transformar
cantidades_originales = {
    "suelo_filas": len(suelo), "suelo_columnas": suelo.shape[1],
    "territorio_filas": len(territorio), "territorio_columnas": territorio.shape[1],
}
print(cantidades_originales)

# Trabajo sobre copias; suelo y territorio no se modifican
analisis = suelo.copy()
analisis.insert(0, "id_fila", range(len(analisis)))   # posición de la fila en esta copia del CSV
terr = territorio.copy()

print("Secuencial único:", analisis[S_SEC].is_unique,
      "| filas con Secuencial repetido:", analisis[S_SEC].duplicated(keep=False).sum())
print("id_fila único:", analisis["id_fila"].is_unique)

In [ ]:
for nombre, df in [("suelo", suelo), ("territorio", territorio)]:
    print("\n====", nombre, df.shape)
    diag = pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "nulos": df.isna().sum(),
        "vacios_tras_strip": df.apply(lambda s: s.str.strip().eq("")).sum(),
        "marcador_ND": df.apply(lambda s: s.str.strip().str.upper().eq("ND")).sum(),
    })
    display(diag)

In [ ]:
# Patrones de formato: cada dígito se reemplaza por 9 para ver la "forma" de los valores
print("--- Fecha de análisis (patrones) ---")
print(suelo[S_FECHA].str.strip().str.replace(r"\d", "9", regex=True).value_counts().head(8))
print("\n--- pH (patrones) ---")
print(suelo[S_PH].str.strip().str.replace(r"\d", "9", regex=True).value_counts().head(10))

print("\n--- Nombres territoriales: espacios externos / repetidos ---")
for nombre, df, cols in [("suelo", suelo, [S_DEP, S_MUN]), ("territorio", territorio, [T_DEP, T_MUN])]:
    for c in cols:
        s = df[c]
        n_externos = (s != s.str.strip()).sum()
        n_repetidos = s.str.contains(r"\s{2,}", regex=True).sum()
        print(nombre + "." + c, "| espacios externos:", n_externos, "| espacios repetidos:", n_repetidos, "| distintos:", s.nunique())

print("\n--- Coordenadas DIVIPOLA (texto original) ---")
display(territorio[[T_LON, T_LAT]].head(8))
print("Valores con coma decimal en longitud:", territorio[T_LON].str.contains(",", regex=False).sum(),
      "| en latitud:", territorio[T_LAT].str.contains(",", regex=False).sum())

**Interpretación:**

Antes de transformar dejé registradas las dimensiones de las dos tablas originales y trabajé sobre copias (`analisis` y `terr`), de modo que `suelo` y `territorio` quedan como evidencia. El diagnóstico separa tres cosas que se parecen pero no son lo mismo: `nulos` (que con `keep_default_na=False` debería ser 0), textos vacíos después de `strip()` y marcadores `ND`. Por eso un campo con 0 nulos no está necesariamente completo: la ausencia está escondida como texto.

Los patrones con dígitos reemplazados por 9 muestran que las fechas y el pH no vienen con un formato uniforme, y la revisión de nombres muestra espacios externos y repetidos, que son la causa típica de uniones fallidas. En DIVIPOLA las coordenadas traen coma decimal, así que conviene convertirlas solo en esas dos columnas y después de ver los ejemplos.

**Límites:** el diagnóstico describe esta copia del archivo; no prueba que los valores sean correctos (una fecha bien formateada puede ser falsa). `Secuencial` no se usa como identificador porque no es único; `id_fila` solo identifica la posición en esta copia y cambiaría si el CSV se reordena.

## 2. Conversión de medidas, fechas y coordenadas · 30 minutos

Crea columnas nuevas. Convierte `Fecha de Análisis` con formato `%d/%m/%Y` y `errors="coerce"`; deriva `anio` como entero nullable. Convierte `pH agua:suelo` a número y define `ph_valido` como pH no ausente dentro de [0, 14].

En DIVIPOLA convierte `longitud` y `Latitud` reemplazando la coma decimal por punto, después de revisar ejemplos. No apliques esa regla a todos los campos. Usa `pd.to_numeric(..., errors="coerce")`. Conserva valores originales y distingue ausencias de textos no vacíos que fallaron al convertir.

**Control:** muestra conteos y hasta cinco ejemplos por fallo. Revisa longitud en [−180, 180] y latitud en [−90, 90]. Un rango válido no demuestra que la ubicación sea correcta. No sustituyas ausencias por cero.

In [ ]:
# ---------- 2.1 Fecha ----------
fecha_txt = analisis[S_FECHA].str.strip()
fecha_ausente = fecha_txt.eq("") | fecha_txt.str.upper().eq("ND")
analisis["fecha_dt"] = pd.to_datetime(fecha_txt, format="%d/%m/%Y", errors="coerce")
analisis["anio"] = analisis["fecha_dt"].dt.year.astype("Int64")
fecha_fallo = analisis["fecha_dt"].isna() & ~fecha_ausente   # texto no vacío que no convirtió

# ---------- 2.2 pH ----------
ph_txt = analisis[S_PH].str.strip()
ph_ausente = ph_txt.eq("") | ph_txt.str.upper().eq("ND")
analisis["ph"] = pd.to_numeric(ph_txt, errors="coerce")
ph_fallo = analisis["ph"].isna() & ~ph_ausente
analisis["ph_valido"] = analisis["ph"].between(0, 14)       # NaN -> False
ph_fuera_rango = analisis["ph"].notna() & ~analisis["ph_valido"]

# ---------- 2.3 Coordenadas DIVIPOLA (coma decimal solo en estas dos columnas) ----------
lon_txt = terr[T_LON].str.strip()
lat_txt = terr[T_LAT].str.strip()
lon_ausente = lon_txt.eq("") | lon_txt.str.upper().eq("ND")
lat_ausente = lat_txt.eq("") | lat_txt.str.upper().eq("ND")
terr["longitud_territorial"] = pd.to_numeric(lon_txt.str.replace(",", ".", regex=False), errors="coerce")
terr["latitud_territorial"] = pd.to_numeric(lat_txt.str.replace(",", ".", regex=False), errors="coerce")
lon_fallo = terr["longitud_territorial"].isna() & ~lon_ausente
lat_fallo = terr["latitud_territorial"].isna() & ~lat_ausente
lon_fuera = terr["longitud_territorial"].notna() & ~terr["longitud_territorial"].between(-180, 180)
lat_fuera = terr["latitud_territorial"].notna() & ~terr["latitud_territorial"].between(-90, 90)

# ---------- 2.4 Control de conversión ----------
control_conversion = pd.DataFrame({
    "total":      [len(analisis)] * 2 + [len(terr)] * 2,
    "ausentes":   [fecha_ausente.sum(), ph_ausente.sum(), lon_ausente.sum(), lat_ausente.sum()],
    "convertidos":[analisis["fecha_dt"].notna().sum(), analisis["ph"].notna().sum(),
                   terr["longitud_territorial"].notna().sum(), terr["latitud_territorial"].notna().sum()],
    "fallos":     [fecha_fallo.sum(), ph_fallo.sum(), lon_fallo.sum(), lat_fallo.sum()],
    "fuera_de_rango": [0, ph_fuera_rango.sum(), lon_fuera.sum(), lat_fuera.sum()],
}, index=["fecha_dt", "ph", "longitud_territorial", "latitud_territorial"])
# Consistencia: ausentes + convertidos + fallos = total
control_conversion["cuadra"] = control_conversion["ausentes"] + control_conversion["convertidos"] + control_conversion["fallos"] == control_conversion["total"]
display(control_conversion)

for etiqueta, mascara, df, col in [
    ("fecha", fecha_fallo, analisis, S_FECHA), ("pH", ph_fallo, analisis, S_PH),
    ("pH fuera de [0,14]", ph_fuera_rango, analisis, S_PH),
    ("longitud", lon_fallo, terr, T_LON), ("latitud", lat_fallo, terr, T_LAT)]:
    print(f"\nEjemplos de fallo - {etiqueta}: {int(mascara.sum())} casos")
    if mascara.sum():
        display(df.loc[mascara, [col]].head(5))

print("\nRango de fechas convertidas:", analisis["fecha_dt"].min(), "->", analisis["fecha_dt"].max())
print("Análisis por año:"); print(analisis["anio"].value_counts(dropna=False).sort_index())
print("Filas conservadas:", len(analisis) == cantidades_originales["suelo_filas"],
      len(terr) == cantidades_originales["territorio_filas"])

**Interpretación:**
Creé columnas nuevas (`fecha_dt`, `anio`, `ph`, `ph_valido` y las coordenadas territoriales) sin tocar las originales ni eliminar filas. Con `errors="coerce"` todo lo que no convierte queda como `NaT`/`NaN`; por eso el control distingue tres grupos que suman el total: **ausentes** (vacío o `ND`), **convertidos** y **fallos** (texto no vacío que no pudo convertirse). Esa separación evita confundir "no había dato" con "había un dato mal escrito", que se trata distinto: lo segundo se puede revisar con la fuente.

`ph_valido` exige pH no ausente y dentro de [0, 14]; un valor numérico fuera de rango (por ejemplo, un pH de dos cifras por un error de digitación) queda convertido en `ph` pero marcado como no válido, de modo que no entra en las medias. La coma decimal se reemplazó únicamente en longitud y latitud de DIVIPOLA, después de ver ejemplos; aplicarla a todos los campos habría dañado otros valores.

**Límites:** un rango válido no demuestra que la ubicación sea correcta (una coordenada puede caer en [−180, 180] y estar en otro municipio). Las ausencias no se rellenaron con cero porque 0 es un valor posible y falso en coordenadas y no tiene sentido físico como pH. Fechas con día/mes intercambiados que sean válidas en ambos órdenes no se pueden detectar con este formato.

## 3. Normalización de nombres y claves · 30 minutos

Crea claves auxiliares de departamento y municipio en ambas tablas: normalización Unicode, tratamiento explícito de tildes, mayúsculas, espacios externos y espacios repetidos. Muestra cada operación con `.str`; conserva el texto original. Describe qué diferencias podrían perderse al normalizar.

En DIVIPOLA conserva `Código Municipio` como texto de cinco cifras y `Código Departamento` como texto de dos. Revisa unicidad de código y de la pareja de nombres normalizados. Detecta claves vacías y colisiones antes de unir.

**Control:** exporta las claves ambiguas. Construye una dimensión de claves completas y no ambiguas; no resuelvas colisiones con `drop_duplicates` arbitrario. Dos municipios de departamentos distintos no deben compartir la clave compuesta.

In [ ]:
# ---------- 3.1 Claves normalizadas (cada operación con .str) ----------
for df, c_dep, c_mun in [(analisis, S_DEP, S_MUN), (terr, T_DEP, T_MUN)]:
    for original, clave in [(c_dep, "clave_dep"), (c_mun, "clave_mun")]:
        df[clave] = (
            df[original]
            .str.normalize("NFKD")                          # separa letra y tilde
            .str.encode("ascii", errors="ignore").str.decode("ascii")   # descarta las marcas diacríticas
            .str.upper()                                    # mayúsculas
            .str.strip()                                    # espacios externos
            .str.replace(r"\s+", " ", regex=True)           # espacios repetidos -> uno
        )
        # vacío o marcador ND = clave incompleta (ausente), no una clave válida
        df[clave] = df[clave].mask(df[clave].isin(["", "ND"]))

display(pd.concat([analisis[[S_MUN, "clave_mun"]].head(5).reset_index(drop=True),
                   terr[[T_MUN, "clave_mun"]].head(5).reset_index(drop=True)], axis=1, keys=["suelo", "territorio"]))

# ---------- 3.2 Códigos DIVIPOLA como texto (5 y 2 cifras) ----------
terr["codigo_departamento"] = terr[T_CDEP].str.strip().str.zfill(2)
terr["codigo_municipio"] = terr[T_CMUN].str.strip().str.zfill(5)
cod_ok = (terr["codigo_municipio"].str.fullmatch(r"\d{5}") & terr["codigo_departamento"].str.fullmatch(r"\d{2}")
          & (terr["codigo_municipio"].str[:2] == terr["codigo_departamento"]))
print("Códigos con formato inválido o incoherentes:", (~cod_ok).sum())
terr.loc[~cod_ok, ["codigo_departamento", "codigo_municipio"]] = pd.NA
terr["nombre_departamento_territorial"] = terr[T_DEP]
terr["nombre_municipio_territorial"] = terr[T_MUN]

# ---------- 3.3 Claves vacías, unicidad y colisiones ----------
incompletas_terr = terr[terr[["clave_dep", "clave_mun", "codigo_municipio"]].isna().any(axis=1)]
base = terr.dropna(subset=["clave_dep", "clave_mun", "codigo_municipio"])
print("Claves territoriales incompletas:", len(incompletas_terr))
print("Claves incompletas en suelo:", analisis[["clave_dep", "clave_mun"]].isna().any(axis=1).sum())

# Duplicados exactos (misma información completa) se colapsan; no es una elección arbitraria
cols_dim = ["codigo_departamento", "codigo_municipio", "clave_dep", "clave_mun",
            "nombre_departamento_territorial", "nombre_municipio_territorial",
            "longitud_territorial", "latitud_territorial"]
n_antes = len(base)
base = base.drop_duplicates(subset=cols_dim)
print("Filas duplicadas exactas colapsadas:", n_antes - len(base))

codigos_por_pareja = base.groupby(["clave_dep", "clave_mun"])["codigo_municipio"].nunique()
parejas_por_codigo = base.groupby("codigo_municipio")[["clave_dep", "clave_mun"]].apply(lambda g: len(g.drop_duplicates()))
pareja_multicodigo = codigos_por_pareja[codigos_por_pareja > 1].index
codigo_multipareja = parejas_por_codigo[parejas_por_codigo > 1].index
codigo_repetido = base.loc[base["codigo_municipio"].duplicated(keep=False), "codigo_municipio"].unique()
print("Parejas de nombres con más de un código:", len(pareja_multicodigo))
print("Códigos con más de una pareja de nombres:", len(codigo_multipareja))
print("Códigos repetidos (filas distintas, mismo código):", len(codigo_repetido))

# ---------- 3.4 Tabla de claves ambiguas (se exporta) ----------
llave_pareja = base.set_index(["clave_dep", "clave_mun"]).index
es_pareja_amb = llave_pareja.isin(pareja_multicodigo)
es_codigo_amb = base["codigo_municipio"].isin(codigo_multipareja) | base["codigo_municipio"].isin(codigo_repetido)
ambiguas = pd.concat([
    incompletas_terr.assign(motivo="clave_incompleta"),
    base[es_pareja_amb].assign(motivo="pareja_con_varios_codigos"),
    base[es_codigo_amb & ~es_pareja_amb].assign(motivo="codigo_repetido_o_con_varias_parejas"),
])[cols_dim + [T_CDEP, T_CMUN, "motivo"]]
display(ambiguas)
ambiguas.to_csv(OUT + "/claves_ambiguas.csv", index=False, encoding="utf-8-sig")

# ---------- 3.5 Dimensión completa y no ambigua ----------
dimension = base[~es_pareja_amb & ~es_codigo_amb][cols_dim].copy()
assert dimension["codigo_municipio"].is_unique
assert not dimension.duplicated(["clave_dep", "clave_mun"]).any()
print("Dimensión:", dimension.shape, "| filas DIVIPOLA:", len(terr))

**Interpretación:**
Normalicé nombres con operaciones `.str` visibles: descomposición Unicode (`NFKD`), eliminación de marcas diacríticas, mayúsculas, `strip()` y compresión de espacios repetidos. Conservé el texto original y guardé el resultado en `clave_dep` y `clave_mun`. Los vacíos y `ND` quedan como ausentes en la clave porque, si no, dos registros sin municipio "coincidirían" entre sí en un `merge`.

Los códigos DIVIPOLA se completaron con ceros a la izquierda (`zfill`) y se validaron: cinco cifras para el municipio, dos para el departamento y que el municipio empiece por su departamento. La unión se hace por la **pareja** (departamento, municipio) y no solo por el municipio, porque nombres como "San José" existen en varios departamentos. Antes de construir la dimensión revisé tres riesgos: parejas de nombres con más de un código, códigos con más de una pareja de nombres y códigos repetidos. Esas filas se exportaron en `claves_ambiguas.csv` y no entran a la dimensión; no usé `drop_duplicates` para escoger una al azar, solo para colapsar copias exactas.

**Límites:** al quitar tildes, `ñ` y `n`, o `á` y `a`, quedan iguales; eso ayuda a unir, pero podría fundir dos nombres que solo difieren por la tilde. Tampoco se detectan errores ortográficos, abreviaturas ni cambios de nombre de un municipio. Mantener fuera las claves ambiguas es conservador: reduce falsos enlaces, pero puede dejar sin enlace análisis que sí pertenecían a un municipio.

## 4. Integración y equivalencias justificadas · 40 minutos

Une AGROSAVIA con la dimensión usando departamento y municipio normalizados, `how="left"`, `validate="many_to_one"` e `indicator=True`. No permitas que claves incompletas coincidan entre sí. Conserva una fila por análisis y distingue coincidencia, clave incompleta, ambigüedad y falta de correspondencia.

Revisa hasta diez nombres sin correspondencia. Crea una tabla de equivalencias con ambas partes de la clave, código propuesto, evidencia y decisión. Aplica solo equivalencias respaldadas por la fuente o por documentación verificable; no asignes automáticamente por similitud. Si no puedes justificar ninguna, entrega la tabla con su diagnóstico. Repite los controles tras aplicarlas y conserva el estado anterior.

**Control:** filas e `id_fila` deben conservarse. Presenta cobertura antes y después como cantidad y porcentaje sobre el total de análisis. No confundas cobertura del enlace con cobertura del territorio.

In [4]:
# ---------- 4.1 Unión: AGROSAVIA (izquierda) con la dimensión ----------
completa = analisis["clave_dep"].notna() & analisis["clave_mun"].notna()   # claves incompletas no entran al merge
cols_enlace = ["codigo_departamento", "codigo_municipio", "nombre_departamento_territorial",
               "nombre_municipio_territorial", "longitud_territorial", "latitud_territorial"]

enlace = analisis.loc[completa, ["id_fila", "clave_dep", "clave_mun"]].merge(
    dimension[["clave_dep", "clave_mun"] + cols_enlace],
    on=["clave_dep", "clave_mun"], how="left", validate="many_to_one", indicator=True
)
enlace["merge_indicador"] = enlace["_merge"].astype(str)
enlace = enlace.drop(columns=["_merge", "clave_dep", "clave_mun"])

analisis_int = analisis.merge(enlace, on="id_fila", how="left", validate="one_to_one")

# ---------- 4.2 Estado de la unión: coincidencia / incompleta / ambigua / sin correspondencia ----------
pares_amb = pd.concat([
    base.loc[es_pareja_amb, ["clave_dep", "clave_mun"]],
    base.loc[es_codigo_amb, ["clave_dep", "clave_mun"]],
]).drop_duplicates()
pares_amb["es_ambigua"] = True
marca_amb = analisis_int[["id_fila", "clave_dep", "clave_mun"]].merge(
    pares_amb, on=["clave_dep", "clave_mun"], how="left", validate="many_to_one")["es_ambigua"].fillna(False).astype(bool).to_numpy()

analisis_int["estado_union"] = np.select(
    [~completa.to_numpy(),
     analisis_int["merge_indicador"].eq("both").to_numpy(),
     marca_amb],
    ["clave_incompleta", "coincide", "ambigua"],
    default="sin_correspondencia",
)
analisis_int["union_via"] = np.where(analisis_int["estado_union"].eq("coincide"), "clave_normalizada", None)
analisis_int["estado_union_previo"] = analisis_int["estado_union"]

assert len(analisis_int) == len(analisis) and analisis_int["id_fila"].equals(analisis["id_fila"])
assert analisis_int["id_fila"].is_unique
n_total = len(analisis_int)
antes = analisis_int["estado_union"].value_counts().rename("cantidad").to_frame()
antes["porcentaje"] = (100 * antes["cantidad"] / n_total).round(2)
print("Cobertura ANTES de equivalencias (sobre el total de análisis):")
display(antes)

# ---------- 4.3 Nombres sin correspondencia ----------
sin_corr = analisis_int[analisis_int["estado_union"].eq("sin_correspondencia")]
top_sin = (sin_corr.groupby(["clave_dep", "clave_mun"]).size().rename("n_analisis")
           .reset_index().sort_values("n_analisis", ascending=False).head(10).reset_index(drop=True))
display(top_sin)

# ---------- 4.4 Tabla de equivalencias (la similitud es solo una pista, no una decisión) ----------
import difflib
pistas = []
for _, r in top_sin.iterrows():
    nombres_dep = dimension.loc[dimension["clave_dep"].eq(r["clave_dep"]), "clave_mun"].tolist()
    c = difflib.get_close_matches(r["clave_mun"], nombres_dep, n=1, cutoff=0.8)
    pistas.append(c[0] if c else pd.NA)

# Solo se aplican equivalencias respaldadas por la fuente o documentación verificable.
# Para aplicar una, agrégala aquí con su evidencia (código de 5 cifras que exista en DIVIPOLA).
EQUIV_VERIFICADAS = {
    # ("DEPARTAMENTO NORMALIZADO", "MUNICIPIO NORMALIZADO"): ("codigo5", "evidencia: fuente/documento"),
}
equivalencias = top_sin.assign(candidato_por_similitud=pistas)
equivalencias["codigo_propuesto"] = [EQUIV_VERIFICADAS.get((d, m), (pd.NA, pd.NA))[0] for d, m in zip(equivalencias["clave_dep"], equivalencias["clave_mun"])]
equivalencias["evidencia"] = [EQUIV_VERIFICADAS.get((d, m), (pd.NA, "sin evidencia verificable: la similitud de texto no basta"))[1] for d, m in zip(equivalencias["clave_dep"], equivalencias["clave_mun"])]
equivalencias["decision"] = np.where(equivalencias["codigo_propuesto"].notna(), "aplicar", "no aplicar")
display(equivalencias)

# ---------- 4.5 Aplicación de las equivalencias respaldadas ----------
assert (analisis_int["id_fila"].to_numpy() == np.arange(len(analisis_int))).all()
eq_apl = equivalencias.loc[equivalencias["decision"].eq("aplicar"), ["clave_dep", "clave_mun", "codigo_propuesto"]]
if len(eq_apl):
    rel = (analisis_int.loc[analisis_int["estado_union"].eq("sin_correspondencia"), ["id_fila", "clave_dep", "clave_mun"]]
           .merge(eq_apl, on=["clave_dep", "clave_mun"], how="inner", validate="many_to_one")
           .merge(dimension[cols_enlace], left_on="codigo_propuesto", right_on="codigo_municipio", how="inner", validate="many_to_one"))
    for c in cols_enlace:
        analisis_int.loc[rel["id_fila"].to_numpy(), c] = rel[c].to_numpy()
    analisis_int.loc[rel["id_fila"].to_numpy(), "estado_union"] = "equivalencia_verificada"
    analisis_int.loc[rel["id_fila"].to_numpy(), "union_via"] = "equivalencia"
    print("Filas resueltas por equivalencia:", len(rel))
else:
    print("No se aplicó ninguna equivalencia: ninguna tiene respaldo verificable.")

# ---------- 4.6 Cobertura antes y después ----------
cobertura = pd.DataFrame({
    "antes": analisis_int["estado_union_previo"].value_counts(),
    "despues": analisis_int["estado_union"].value_counts(),
}).fillna(0).astype(int)
cobertura["pct_antes"] = (100 * cobertura["antes"] / n_total).round(2)
cobertura["pct_despues"] = (100 * cobertura["despues"] / n_total).round(2)
display(cobertura)
enlazados = analisis_int["codigo_municipio"].notna()
print(f"Análisis con municipio asociado: {enlazados.sum()} de {n_total} ({100*enlazados.mean():.2f} %)")
print("Filas e id_fila conservados:", len(analisis_int) == n_total == cantidades_originales["suelo_filas"])
print("Cobertura del enlace (análisis con código). No es cobertura del territorio: municipios con al menos un análisis =",
      analisis_int["codigo_municipio"].nunique(), "de", dimension["codigo_municipio"].nunique(), "en la dimensión")

NameError: name 'analisis' is not defined

**Interpretación:**
La unión es un `merge` `how="left"` con `validate="many_to_one"` e `indicator=True`: muchas filas de AGROSAVIA pueden apuntar a un mismo municipio, pero cada pareja de nombres debe apuntar a una sola fila de la dimensión. Dejé fuera del `merge` las filas con clave incompleta (pandas, por defecto, hace coincidir `NaN` con `NaN`) y luego recuperé cada fila por `id_fila`, de modo que el resultado tiene exactamente una fila por análisis. El estado final distingue cuatro situaciones: `coincide`, `clave_incompleta`, `ambigua` (la pareja existe pero apunta a más de un código) y `sin_correspondencia`.

En la tabla de equivalencias usé `difflib` solo para sugerir candidatos, porque una similitud de texto no demuestra que dos nombres sean el mismo municipio; la decisión queda en "no aplicar" mientras no exista evidencia verificable (por ejemplo, el código en la fuente o un acto administrativo de cambio de nombre). Al agregar una equivalencia respaldada en `EQUIV_VERIFICADAS`, el resultado se recalcula y se conserva el estado anterior en `estado_union_previo`, así que la cobertura antes y después es comparable.

**Límites:** la cobertura es del **enlace** (cuántos análisis quedaron con código) y no del territorio (cuántos municipios tienen datos). Que un nombre coincida no garantiza que el municipio sea el correcto para la fecha del análisis, y DIVIPOLA es un corte de 2024.

## 5. Resumen territorial y límites · 25 minutos

Agrupa los registros asociados por código municipal. Calcula total de análisis, cantidad de pH válidos y media de pH válido. Conserva los municipios sin pH válido con media ausente. Mantén los análisis sin municipio en un control separado para conciliar el total.

**Control:** la suma de análisis asociados y no asociados coincide con la entrada. Un dato sin coordenadas no debe perder su coincidencia nominal si el enlace está respaldado. Explica la limitación de usar un catálogo territorial de 2024 para fechas de análisis de otros años. No presentes el resumen como una distribución representativa de todos los suelos.

In [5]:
# ---------- 5.1 Resumen por municipio ----------
analisis_int["ph_para_media"] = analisis_int["ph"].where(analisis_int["ph_valido"])   # solo pH válido entra a la media
asociados = analisis_int[analisis_int["codigo_municipio"].notna()]
no_asociados = analisis_int[analisis_int["codigo_municipio"].isna()]

resumen_mun = (asociados.groupby("codigo_municipio", as_index=False)
               .agg(total_analisis=("id_fila", "size"),
                    n_ph_validos=("ph_valido", "sum"),
                    ph_media_valido=("ph_para_media", "mean")))
resumen_mun = resumen_mun.merge(dimension[["codigo_municipio", "nombre_departamento_territorial", "nombre_municipio_territorial"]],
                                on="codigo_municipio", how="left", validate="one_to_one")
display(resumen_mun.sort_values("total_analisis", ascending=False).head(10))
print("Municipios con 0 pH válidos (media ausente):", resumen_mun["ph_media_valido"].isna().sum())

# ---------- 5.2 Control separado de análisis sin municipio ----------
control_no_asoc = no_asociados["estado_union"].value_counts().rename("analisis").to_frame()
display(control_no_asoc)

# ---------- 5.3 Conciliación del total ----------
conciliacion = {
    "analisis_asociados (suma resumen)": int(resumen_mun["total_analisis"].sum()),
    "analisis_no_asociados": len(no_asociados),
    "total_entrada": len(suelo),
}
print(conciliacion)
assert conciliacion["analisis_asociados (suma resumen)"] + conciliacion["analisis_no_asociados"] == conciliacion["total_entrada"]

# Un dato sin coordenadas conserva su coincidencia nominal
sin_coord = asociados["longitud_territorial"].isna() | asociados["latitud_territorial"].isna()
print("Asociados sin coordenadas territoriales (siguen asociados):", int(sin_coord.sum()))

# ---------- 5.4 Hallazgos con cifras de esta ejecución ----------
top = resumen_mun.sort_values("total_analisis", ascending=False).iloc[0]
con_ph = resumen_mun.dropna(subset=["ph_media_valido"])
print(f"1) {len(resumen_mun)} municipios reúnen {conciliacion['analisis_asociados (suma resumen)']} análisis; el mayor es "
      f"{top['nombre_municipio_territorial']} con {int(top['total_analisis'])} ({100*top['total_analisis']/n_total:.1f} % del total).")
print(f"2) La media de pH válido por municipio va de {con_ph['ph_media_valido'].min():.2f} a {con_ph['ph_media_valido'].max():.2f} "
      f"(sobre {int(resumen_mun['n_ph_validos'].sum())} mediciones válidas de {int(asociados['ph'].notna().sum())} convertidas).")
print(f"3) {len(no_asociados)} análisis ({100*len(no_asociados)/n_total:.1f} %) no tienen municipio y quedan fuera del resumen; "
      f"años en los datos: {sorted(analisis_int['anio'].dropna().unique().tolist())} frente a un catálogo de 2024.")

NameError: name 'analisis_int' is not defined

**Interpretación:**
El resumen agrupa solo los análisis asociados por `codigo_municipio` y calcula tres medidas: total de análisis, cantidad de pH válidos y media del pH válido. La media usa una columna auxiliar (`ph_para_media`) con el pH solo donde `ph_valido` es verdadero; los municipios sin pH válido conservan la media ausente (`NaN`) en lugar de convertirse en 0. Los análisis sin municipio quedan en un control aparte por estado de unión y se concilian con el total de entrada: asociados más no asociados es igual a las filas del CSV. Además, los análisis asociados que no tienen coordenadas siguen asociados, porque la coincidencia nominal es independiente de la geolocalización.

**Tres hallazgos (ver cifras en la salida anterior), con sus límites:**
1. *Concentración:* unos pocos municipios reúnen buena parte de los análisis asociados. Esto refleja dónde se tomaron las muestras, no dónde hay más suelo agrícola.
2. *pH:* la media por municipio solo describe las observaciones disponibles; con pocos pH válidos, una sola medición mueve la media, así que hay que leerla junto con `n_ph_validos`.
3. *Cobertura:* los análisis sin municipio (clave incompleta, ambigua o sin correspondencia) quedan fuera; si fueran de zonas específicas, el resumen estaría sesgado.

**Límite temporal:** DIVIPOLA es un corte de 2024 y los análisis son de otras fechas. Un municipio creado, fusionado o renombrado después de la fecha del análisis podría no coincidir o coincidir con un nombre que entonces significaba otro territorio. Por último, el resumen no es una distribución representativa de todos los suelos: las muestras no son aleatorias ni uniformes.

## 6. Lectura espacial, CRS y calidad geométrica · 30 minutos

Ejecuta las dos lecturas directas siguientes. Inspecciona columnas, tipos geométricos, CRS y extensión (`total_bounds`). Cuenta por separado geometrías ausentes, vacías e inválidas. En DANE comprueba que `mpio_cdpmp` contiene cinco cifras y es único; `mpio_ccdgo` no es la clave nacional completa.

Conserva originales y trabaja sobre copias. Separa ausentes/vacías y registra sus códigos. Si hay geometrías inválidas, muestra una reparación con `.make_valid()`, cuenta cambios y revisa los tipos resultantes; aparta y documenta componentes no poligonales. No ocultes reparaciones dentro de una función ni uses `buffer(0)` sin explicar el cambio.

**Control:** se esperan 1.122 entidades DANE y cinco elementos de la muestra IGAC. Si tu copia difiere, revisa la fuente y documenta la diferencia. Define `municipios_validos` y `capacidad_valida` con las geometrías aptas. Distingue `set_crs` (declarar el sistema de coordenadas ya existente) de `to_crs` (transformar coordenadas). Si falta el CRS, revisa los metadatos antes de asignarlo.

In [6]:
municipios = gpd.read_file(
    "zip://" + RAW + "/dane_municipios.zip",
    engine="pyogrio"
)
display(municipios.head())
print(municipios.shape, municipios.crs)
print(municipios.columns.tolist())

,dpto_ccdgo,mpio_ccdgo,mpio_cdpmp,dpto_cnmbr,mpio_cnmbr,mpio_crslc,mpio_tipo,mpio_narea,mpio_nano,shape_Leng,shape_Area,geometry
0,05,001,05001,ANTIOQUIA,MEDELLÍN,1965,MUNICIPIO,374.741484,2025,1.035380,0.030608,"POLYGON ((-75.4857 6.20163, -75.48567 6.20148,..."
1,05,002,05002,ANTIOQUIA,ABEJORRAL,1814,MUNICIPIO,506.959782,2024,1.158504,0.041384,"POLYGON ((-75.46938 5.94575, -75.46897 5.94571..."
2,05,004,05004,ANTIOQUIA,ABRIAQUÍ,1912,MUNICIPIO,296.974188,2025,0.810667,0.024253,"POLYGON ((-76.09047 6.74542, -76.09066 6.74548..."
3,05,021,05021,ANTIOQUIA,ALEJANDRÍA,Decreto departamental 304 de 1907,MUNICIPIO,128.856431,2024,0.705200,0.010535,"POLYGON ((-75.0332 6.41586, -75.03313 6.41585,..."
4,05,030,05030,ANTIOQUIA,AMAGÁ,1912,MUNICIPIO,84.117145,2024,0.445533,0.006867,"POLYGON ((-75.67587 6.08561, -75.6754 6.08491,..."


(1122, 12) EPSG:4326
['dpto_ccdgo', 'mpio_ccdgo', 'mpio_cdpmp', 'dpto_cnmbr', 'mpio_cnmbr', 'mpio_crslc', 'mpio_tipo', 'mpio_narea', 'mpio_nano', 'shape_Leng', 'shape_Area', 'geometry']


In [7]:
capacidad = gpd.read_file(
    RAW + "/igac_capacidad.json",
    engine="pyogrio"
)
display(capacidad.head())
print(capacidad.shape, capacidad.crs)
print(capacidad.columns.tolist())

,OBJECTID,UCCapacida,Area,Descripcio,Shape_Length,Shape_Area,geometry
0,1,2c,1599.710606,Capacidad alta para agricultura,61943.688570,1.599711e+07,"POLYGON ((-74.22594 4.55804, -74.22514 4.56111..."
1,2,2c,359.860856,Capacidad alta para agricultura,11388.723446,3.598609e+06,"POLYGON ((-74.21308 4.60131, -74.21388 4.60123..."
2,3,2c,1290.315155,Capacidad alta para agricultura,34986.556763,1.290315e+07,"POLYGON ((-74.25492 4.62867, -74.25572 4.62956..."
3,4,2c,877.464418,Capacidad alta para agricultura,24177.793654,8.774644e+06,"POLYGON ((-74.15084 4.66405, -74.15155 4.66429..."
4,5,2c,423.111236,Capacidad alta para agricultura,12919.093320,4.231112e+06,"POLYGON ((-74.15661 4.68019, -74.15822 4.68213..."


(5, 7) EPSG:4326
['OBJECTID', 'UCCapacida', 'Area', 'Descripcio', 'Shape_Length', 'Shape_Area', 'geometry']


In [8]:
import shapely

# ---------- 6.1 Inspección de las dos capas ----------
for nombre, g in [("DANE municipios", municipios), ("IGAC capacidad", capacidad)]:
    vacia = g.geometry.notna() & g.geometry.is_empty
    presente = g.geometry.notna() & ~g.geometry.is_empty
    print("\n===", nombre, "===")
    print("Filas:", len(g), "| CRS:", g.crs, "| EPSG:", g.crs.to_epsg() if g.crs is not None else None)
    print("Extensión total_bounds (minx, miny, maxx, maxy):", g.total_bounds)
    print("Columnas y tipos:"); print(g.dtypes.astype(str).to_dict())
    print("Tipos geométricos:", g.geom_type.value_counts(dropna=False).to_dict())
    print("Ausentes:", int(g.geometry.isna().sum()), "| Vacías:", int(vacia.sum()),
          "| Inválidas:", int((presente & ~g.geometry.is_valid).sum()))

# Claves DANE
cod = municipios["mpio_cdpmp"].astype(str)
print("\nmpio_cdpmp: 5 cifras en todos =", bool(cod.str.fullmatch(r"\d{5}").all()),
      "| único =", municipios["mpio_cdpmp"].is_unique, "| duplicados =", int(municipios["mpio_cdpmp"].duplicated().sum()))
print("mpio_ccdgo único (no es la clave nacional completa):", municipios["mpio_ccdgo"].is_unique)
print("Control de tamaño -> DANE:", len(municipios), "(esperado 1122) | IGAC:", len(capacidad), "(esperado 5)")
print("Mismo CRS en ambas capas:", municipios.crs == capacidad.crs)
if municipios.crs is None or capacidad.crs is None:
    raise ValueError("Falta el CRS en alguna capa: revisa los metadatos antes de usar set_crs.")

# ---------- 6.2 Copias de trabajo: apartar ausentes/vacías ----------
muni_trab = municipios.copy()
aus = muni_trab.geometry.isna()
vac = muni_trab.geometry.notna() & muni_trab.geometry.is_empty
muni_sin_geom = muni_trab.loc[aus | vac, ["mpio_cdpmp"]].assign(motivo=np.where(aus[aus | vac], "ausente", "vacia"))
print("\nMunicipios sin geometría utilizable (códigos registrados):"); display(muni_sin_geom)
presentes = muni_trab.loc[~(aus | vac)].copy()

# ---------- 6.3 Reparación visible de geometrías inválidas ----------
invalidas = ~presentes.geometry.is_valid
print("Inválidas antes de reparar:", int(invalidas.sum()), "->", presentes.loc[invalidas, "mpio_cdpmp"].tolist())
reparada = presentes.geometry.make_valid()
cambiadas = ~presentes.geometry.geom_equals_exact(reparada, tolerance=0)
print("Geometrías modificadas por make_valid:", int(cambiadas.sum()))
print("Tipos resultantes de las reparadas:", reparada[invalidas].geom_type.value_counts().to_dict())
print("Área (ha, medida en EPSG:9377) antes/después de reparar las inválidas:")
display(pd.DataFrame({"antes_ha": presentes.geometry[invalidas].to_crs(epsg=9377).area / 10000,
                      "despues_ha": reparada[invalidas].to_crs(epsg=9377).area / 10000}))
presentes["geometry"] = reparada
print("Válidas después:", bool(presentes.geometry.is_valid.all()))

# Componentes no poligonales: extraer las poligonales de las colecciones y documentar el resto
registro_no_pol = []
tipos = presentes.geom_type
for idx in presentes.index[tipos.eq("GeometryCollection")]:
    partes = shapely.get_parts(presentes.at[idx, "geometry"])
    pol = [p for p in partes if p.geom_type in ("Polygon", "MultiPolygon")]
    otras = [p for p in partes if p.geom_type not in ("Polygon", "MultiPolygon")]
    registro_no_pol.append({"mpio_cdpmp": presentes.at[idx, "mpio_cdpmp"], "partes_poligonales": len(pol),
                            "componentes_no_poligonales": len(otras), "tipos": sorted({p.geom_type for p in otras})})
    if pol:
        presentes.at[idx, "geometry"] = shapely.union_all(pol)
registro_no_pol = pd.DataFrame(registro_no_pol)
print("Colecciones con componentes no poligonales (documentadas):"); display(registro_no_pol)

es_pol = presentes.geom_type.isin(["Polygon", "MultiPolygon"])
municipios_apartados = presentes[~es_pol]
municipios_validos = presentes[es_pol & presentes.geometry.is_valid].copy()
print("municipios_validos:", len(municipios_validos), "| apartados no poligonales:", len(municipios_apartados),
      "| sin geometría:", len(muni_sin_geom))
assert len(municipios_validos) + len(municipios_apartados) + len(muni_sin_geom) == len(municipios)

# ---------- 6.4 Capacidad IGAC ----------
capacidad_trab = capacidad.copy()
capacidad_trab["id_igac"] = range(len(capacidad_trab))        # posición de la fila original
aus_c = capacidad_trab.geometry.isna()
vac_c = capacidad_trab.geometry.notna() & capacidad_trab.geometry.is_empty
cap_presentes = capacidad_trab.loc[~(aus_c | vac_c)].copy()
inv_c = ~cap_presentes.geometry.is_valid
print("IGAC inválidas:", int(inv_c.sum()))
cap_presentes["geometry"] = cap_presentes.geometry.make_valid()
capacidad_valida = cap_presentes[cap_presentes.geom_type.isin(["Polygon", "MultiPolygon"]) & cap_presentes.geometry.is_valid].copy()
print("capacidad_valida:", len(capacidad_valida), "de", len(capacidad))


=== DANE municipios ===
Filas: 1122 | CRS: EPSG:4326 | EPSG: 4326
Extensión total_bounds (minx, miny, maxx, maxy): [-81.73562056  -4.22940647 -66.84721543  13.39472776]
Columnas y tipos:
{'dpto_ccdgo': 'object', 'mpio_ccdgo': 'object', 'mpio_cdpmp': 'object', 'dpto_cnmbr': 'object', 'mpio_cnmbr': 'object', 'mpio_crslc': 'object', 'mpio_tipo': 'object', 'mpio_narea': 'float64', 'mpio_nano': 'int32', 'shape_Leng': 'float64', 'shape_Area': 'float64', 'geometry': 'geometry'}
Tipos geométricos: {'Polygon': 1113, 'MultiPolygon': 9}
Ausentes: 0 | Vacías: 0 | Inválidas: 0

=== IGAC capacidad ===
Filas: 5 | CRS: EPSG:4326 | EPSG: 4326
Extensión total_bounds (minx, miny, maxx, maxy): [-74.289826     4.48683424 -74.13713915   4.68608354]
Columnas y tipos:
{'OBJECTID': 'int32', 'UCCapacida': 'object', 'Area': 'float64', 'Descripcio': 'object', 'Shape_Length': 'float64', 'Shape_Area': 'float64', 'geometry': 'geometry'}
Tipos geométricos: {'Polygon': 5}
Ausentes: 0 | Vacías: 0 | Inválidas: 0

mpio_

,mpio_cdpmp,motivo


Inválidas antes de reparar: 0 -> []
Geometrías modificadas por make_valid: 0
Tipos resultantes de las reparadas: {}
Área (ha, medida en EPSG:9377) antes/después de reparar las inválidas:


,antes_ha,despues_ha


Válidas después: True
Colecciones con componentes no poligonales (documentadas):


""


municipios_validos: 1122 | apartados no poligonales: 0 | sin geometría: 0
IGAC inválidas: 0
capacidad_valida: 5 de 5


**Interpretación:**
Leí las dos capas con `gpd.read_file(..., engine="pyogrio")` y revisé columnas, tipos geométricos, CRS y extensión. Ausente, vacía e inválida son tres problemas distintos y se cuentan por separado: una geometría ausente (`None`) o vacía no se puede reparar, solo apartar y registrar por código; una inválida (por ejemplo, un polígono que se cruza consigo mismo) sí se puede reparar con `make_valid()`. Mostré la reparación a la vista, conté las geometrías cambiadas, revisé los tipos resultantes (una reparación puede devolver `MultiPolygon` o `GeometryCollection`) y comparé el área antes y después. No usé `buffer(0)`, porque puede cambiar o perder partes de la geometría sin avisar.

La clave de DANE es `mpio_cdpmp` (cinco cifras y única); `mpio_ccdgo` solo identifica el municipio dentro del departamento y por eso no es única a nivel nacional. Sobre el CRS: `set_crs` **declara** el sistema que ya tienen las coordenadas (solo cuando falta el metadato y se confirmó en la documentación), mientras que `to_crs` **transforma** las coordenadas a otro sistema. Aquí ambas capas ya traen CRS, así que no fue necesario declararlo; si faltara, el código se detiene en vez de asignarlo en silencio.

**Límites:** el conteo esperado (1.122 y 5) es un control, no una garantía de calidad. Si mi copia difiere, debo revisar la fuente y documentarlo en vez de ajustar los datos. Una geometría válida no implica que esté bien ubicada ni actualizada (MGN 2025), y reparar altera la forma original aunque el cambio sea pequeño.

## 7. Integración de tabla y polígonos, mapa y área · 30 minutos

Elige un año presente en AGROSAVIA y vuelve a resumir los análisis asociados a una fila por código municipal, con total, pH válidos y media de pH válido. Conserva el año elegido en el resultado.

Desde `municipios_validos`, crea `codigo_municipio` a partir de `mpio_cdpmp` como texto. Comprueba ambas claves y usa `municipios_validos.merge(resumen_anual, on="codigo_municipio", how="left", validate="one_to_one", indicator=True)`. Guarda el resultado en `mapa_municipal`. Mantén la geometría activa y el CRS. Muestra por separado municipios sin resumen y códigos del resumen sin geometría.

Transforma una copia con `.to_crs(epsg=9377)` y calcula `area_ha = geometry.area / 10000`. Explica por qué no calculas hectáreas directamente en grados. Crea un mapa del departamento que elijas, con `GeoDataFrame.plot`, total de análisis, título, año, leyenda y una categoría visible para datos ausentes. Guarda `mapa_analisis.png` con `plt.savefig`.

**Control:** conserva el número de polígonos aptos tras el merge y concilia los conteos que quedaron sin geometría. La ausencia de registros no demuestra cero análisis en el territorio. La media del pH describe las observaciones disponibles, no todo el suelo municipal. Documenta la diferencia entre DIVIPOLA 2024 y MGN 2025.

In [9]:
# ---------- 7.1 Año elegido y resumen anual ----------
print(analisis_int.loc[analisis_int["codigo_municipio"].notna(), "anio"].value_counts().sort_index())
anio_elegido = int(analisis_int.loc[analisis_int["codigo_municipio"].notna(), "anio"].value_counts().idxmax())
print("Año elegido (el de más análisis asociados):", anio_elegido)

resumen_anual = (analisis_int[analisis_int["codigo_municipio"].notna() & analisis_int["anio"].eq(anio_elegido)]
                 .groupby("codigo_municipio", as_index=False)
                 .agg(total_analisis=("id_fila", "size"), n_ph_validos=("ph_valido", "sum"),
                      ph_media_valido=("ph_para_media", "mean")))
resumen_anual.insert(1, "anio", anio_elegido)

# ---------- 7.2 Merge tabla + polígonos ----------
municipios_validos["codigo_municipio"] = municipios_validos["mpio_cdpmp"].astype(str).str.strip()
print("Claves únicas -> polígonos:", municipios_validos["codigo_municipio"].is_unique,
      "| resumen:", resumen_anual["codigo_municipio"].is_unique,
      "| dtypes:", municipios_validos["codigo_municipio"].dtype, resumen_anual["codigo_municipio"].dtype)

mapa_municipal = municipios_validos.merge(resumen_anual, on="codigo_municipio", how="left",
                                          validate="one_to_one", indicator=True)
mapa_municipal["merge_resumen"] = mapa_municipal["_merge"].astype(str)
mapa_municipal = mapa_municipal.drop(columns="_merge")
print(type(mapa_municipal).__name__, "| geometría activa:", mapa_municipal.geometry.name, "| CRS:", mapa_municipal.crs)
print("Polígonos aptos antes/después del merge:", len(municipios_validos), len(mapa_municipal))
assert len(mapa_municipal) == len(municipios_validos)

sin_resumen = mapa_municipal[mapa_municipal["merge_resumen"].eq("left_only")]
codigos_sin_geom = sorted(set(resumen_anual["codigo_municipio"]) - set(municipios_validos["codigo_municipio"]))
print("Municipios (polígonos) sin resumen:", len(sin_resumen))
print("Códigos del resumen sin geometría:", codigos_sin_geom,
      "-> análisis que quedan sin polígono:", int(resumen_anual.loc[resumen_anual["codigo_municipio"].isin(codigos_sin_geom), "total_analisis"].sum()))
print("Conciliación del año", anio_elegido, ": resumen =", int(resumen_anual["total_analisis"].sum()),
      "| en el mapa =", int(mapa_municipal["total_analisis"].sum()))

# ---------- 7.3 Área en proyección métrica ----------
mapa_proyectado = mapa_municipal.to_crs(epsg=9377)
mapa_municipal["area_ha"] = mapa_proyectado.geometry.area / 10000
print(mapa_municipal["area_ha"].describe())

# ---------- 7.4 Mapa del departamento con más análisis ----------
from matplotlib.patches import Patch
dep_elegido = (resumen_anual.assign(cod_dep=resumen_anual["codigo_municipio"].str[:2])
               .groupby("cod_dep")["total_analisis"].sum().idxmax())
nom_dep = dimension.loc[dimension["codigo_departamento"].eq(dep_elegido), "nombre_departamento_territorial"].iloc[0]
mapa_dep = mapa_municipal[mapa_municipal["codigo_municipio"].str[:2].eq(dep_elegido)]

fig, ax = plt.subplots(figsize=(8, 8))
mapa_dep.plot(column="total_analisis", cmap="YlGnBu", edgecolor="white", linewidth=0.4, legend=True,
              legend_kwds={"label": "Total de análisis asociados", "shrink": 0.6},
              missing_kwds={"color": "lightgrey", "edgecolor": "white"}, ax=ax)
ax.legend(handles=[Patch(facecolor="lightgrey", edgecolor="white", label="Sin análisis asociados (no implica cero)")], loc="upper center", bbox_to_anchor=(0.5, 0.0))
ax.set_title(f"Análisis de suelo asociados por municipio - {nom_dep} ({anio_elegido})")
ax.set_axis_off()
plt.savefig(OUT + "/mapa_analisis.png", dpi=150, bbox_inches="tight")
plt.show()

NameError: name 'analisis_int' is not defined

**Interpretación:**
Elegí el año con más análisis asociados y lo conservé en la columna `anio` del resumen. Luego creé `codigo_municipio` a partir de `mpio_cdpmp` como texto, comprobé que ambas claves son únicas y del mismo tipo, y unir con `merge(..., how="left", validate="one_to_one", indicator=True)` dejó el mismo número de polígonos aptos. Mantuve visibles los dos controles: municipios sin resumen (polígonos que quedan en gris) y códigos del resumen sin geometría (análisis que no se pueden dibujar). La suma del año en el resumen y en el mapa difiere exactamente en esos últimos.

El área se calcula tras `to_crs(epsg=9377)` (MAGNA-SIRGAS / Origen-Nacional, en metros) dividiendo entre 10.000 para obtener hectáreas. No lo hice directamente en grados porque la unidad es angular: un grado no tiene una longitud constante (los grados de longitud se acortan hacia los polos), así que `geometry.area` daría "grados cuadrados", sin significado en hectáreas, y con distorsión según la latitud.

**Límites:** un municipio en gris significa "sin registros en esta muestra y año", no "cero análisis" en el territorio. La media de pH describe las observaciones disponibles, no todo el suelo del municipio. Además hay una brecha de fuentes: DIVIPOLA es de 2024 y los polígonos MGN son de 2025, de modo que un cambio de código o de límites entre ambos cortes puede dejar municipios sin enlace aunque el nombre exista.

## 8. Intersección espacial con la muestra IGAC · 30 minutos

Proyecta copias de `municipios_validos` y `capacidad_valida` a EPSG:9377. Conserva una columna que identifique cada polígono IGAC a partir de su fila original. Selecciona solo códigos, identificador, atributos IGAC necesarios y geometría; evita nombres de columnas duplicados.

Ejecuta directamente `gpd.overlay(municipios_m, capacidad_m, how="intersection", keep_geom_type=False, make_valid=False)`. `make_valid=False` exige que hayas resuelto la validez de forma visible. Inspecciona los tipos resultantes, registra contactos de área cero y conserva las piezas poligonales de área positiva en `intersecciones`. Calcula hectáreas a partir de la geometría de cada pieza, no del atributo de área original.

Resume el área intersectada por municipio como `area_intersectada_ha` e intégrala a `mapa_municipal` con `validate="one_to_one"`. Para comparar geometrías originales y piezas, muestra un mapa de la extensión de la muestra y guárdalo como `mapa_interseccion.png`. No sumes áreas como cobertura única si hay solapamientos: diagnostica ese riesgo y explica el significado de la suma.

**Control:** presenta cantidad y área de piezas. Contrasta por polígono IGAC el área original y la suma intersectada, con tolerancia explícita y sin exigir igualdad si parte queda fuera de los municipios. Cinco polígonos son una muestra: no permiten estimar la capacidad de uso de un municipio completo. Esta intersección tampoco asigna una clase IGAC a una muestra individual AGROSAVIA.

In [ ]:
# ---------- 8.1 Copias proyectadas con columnas mínimas ----------
print("Atributos IGAC disponibles:", [c for c in capacidad_valida.columns if c not in ("geometry", "id_igac")])
# Solo la clase de capacidad y su descripción; Area y Shape_Area vienen de la fuente y se RECALCULAN tras intersectar
igac_attrs = [c for c in ["UCCapacida", "Descripcio"] if c in capacidad_valida.columns]
print("Atributos IGAC que se conservan:", igac_attrs)
municipios_m = municipios_validos[["codigo_municipio", "geometry"]].to_crs(epsg=9377)
capacidad_m = (capacidad_valida[["id_igac"] + igac_attrs + ["geometry"]]
               .rename(columns={c: "igac_" + c for c in igac_attrs}).to_crs(epsg=9377))
print("Columnas sin duplicados:", municipios_m.columns.tolist(), capacidad_m.columns.tolist())
print("Validez tras proyectar -> municipios:", bool(municipios_m.is_valid.all()), "| capacidad:", bool(capacidad_m.is_valid.all()))

# ---------- 8.2 Overlay (make_valid=False: la validez ya se resolvió en la sección 6) ----------
inter_cruda = gpd.overlay(municipios_m, capacidad_m, how="intersection", keep_geom_type=False, make_valid=False)
print("Resultados crudos:", len(inter_cruda), "| tipos:", inter_cruda.geom_type.value_counts().to_dict())

# Extraer las partes poligonales de las colecciones (si las hay)
for idx in inter_cruda.index[inter_cruda.geom_type.eq("GeometryCollection")]:
    pol = [p for p in shapely.get_parts(inter_cruda.at[idx, "geometry"]) if p.geom_type in ("Polygon", "MultiPolygon")]
    inter_cruda.at[idx, "geometry"] = shapely.union_all(pol) if pol else shapely.Polygon()

# ---------- 8.3 Contactos de área cero y piezas poligonales ----------
AREA_MIN_M2 = 1e-6          # tolerancia explícita: por debajo es un contacto, no una pieza
inter_cruda["area_m2"] = inter_cruda.geometry.area
es_pol = inter_cruda.geom_type.isin(["Polygon", "MultiPolygon"])
contactos = inter_cruda[~es_pol | (inter_cruda["area_m2"] <= AREA_MIN_M2)]
intersecciones = inter_cruda[es_pol & (inter_cruda["area_m2"] > AREA_MIN_M2)].copy()
intersecciones["area_ha"] = intersecciones.geometry.area / 10000          # desde la geometría de cada pieza
print("Contactos / piezas de área ~0 registrados:", len(contactos), "| piezas poligonales:", len(intersecciones))
print("Área total de las piezas (ha):", round(intersecciones["area_ha"].sum(), 2))

# ---------- 8.4 Resumen por municipio e integración a mapa_municipal ----------
area_mun = (intersecciones.groupby("codigo_municipio", as_index=False)
            .agg(area_intersectada_ha=("area_ha", "sum"), n_piezas=("area_ha", "size")))
mapa_municipal = mapa_municipal.merge(area_mun, on="codigo_municipio", how="left", validate="one_to_one")
mapa_municipal[["area_intersectada_ha", "n_piezas"]] = mapa_municipal[["area_intersectada_ha", "n_piezas"]].fillna(0)   # 0 = sin intersección con la muestra
display(mapa_municipal.loc[mapa_municipal["area_intersectada_ha"] > 0, ["codigo_municipio", "area_ha", "area_intersectada_ha", "n_piezas"]])

# ---------- 8.5 Control por polígono IGAC y diagnóstico de solapamientos ----------
TOL_HA = 1e-3
orig = capacidad_m.assign(area_original_ha=capacidad_m.geometry.area / 10000)[["id_igac", "area_original_ha"]]
suma = intersecciones.groupby("id_igac")["area_ha"].sum().rename("suma_intersectada_ha")
control_igac = orig.merge(suma, on="id_igac", how="left", validate="one_to_one").fillna({"suma_intersectada_ha": 0})
control_igac["dif_ha"] = control_igac["area_original_ha"] - control_igac["suma_intersectada_ha"]
control_igac["estado"] = np.select(
    [control_igac["dif_ha"].abs() <= TOL_HA, control_igac["dif_ha"] > TOL_HA],
    ["completo (dentro de municipios)", "parcial (parte fuera de los municipios)"],
    default="la suma excede el área original: revisar solapamientos")
display(control_igac)

# Si la suma de áreas supera el área de la unión, hay solapamiento y la suma NO es cobertura única
suma_piezas = intersecciones["area_ha"].sum()
area_union = intersecciones.geometry.union_all().area / 10000
suma_cap = capacidad_m.geometry.area.sum() / 10000
area_union_cap = capacidad_m.geometry.union_all().area / 10000
print(f"Piezas: suma={suma_piezas:.2f} ha, unión={area_union:.2f} ha, solapamiento={suma_piezas - area_union:.2f} ha")
print(f"Polígonos IGAC: suma={suma_cap:.2f} ha, unión={area_union_cap:.2f} ha, solapamiento={suma_cap - area_union_cap:.2f} ha")

# ---------- 8.6 Mapa de la extensión de la muestra ----------
minx, miny, maxx, maxy = capacidad_m.total_bounds
ctx = municipios_m.cx[minx:maxx, miny:maxy]
fig, ax = plt.subplots(figsize=(8, 8))
ctx.plot(ax=ax, facecolor="none", edgecolor="grey", linewidth=0.6)
capacidad_m.plot(ax=ax, facecolor="none", edgecolor="tab:blue", linewidth=1.8)
intersecciones.plot(ax=ax, facecolor="tab:orange", edgecolor="darkred", alpha=0.6, linewidth=0.5)
ax.set_xlim(minx - 5000, maxx + 5000); ax.set_ylim(miny - 5000, maxy + 5000)
ax.legend(handles=[Patch(facecolor="none", edgecolor="grey", label="Municipios (DANE)"),
                   Patch(facecolor="none", edgecolor="tab:blue", label="Polígonos IGAC originales"),
                   Patch(facecolor="tab:orange", edgecolor="darkred", alpha=0.6, label="Piezas de intersección")], loc="upper center", bbox_to_anchor=(0.5, -0.12), ncol=3, fontsize=8)
ax.set_title("Muestra IGAC e intersección con municipios (EPSG:9377)")
plt.savefig(OUT + "/mapa_interseccion.png", dpi=150, bbox_inches="tight")
plt.show()

**Interpretación:**
`merge` y `overlay` responden preguntas distintas. El `merge` une **atributos por una clave** (el código municipal), sin mirar dónde están las geometrías. El `overlay` usa la **posición y la forma**: calcula, geométricamente, qué parte de cada municipio coincide con cada polígono IGAC y devuelve nuevas piezas con atributos de ambos lados. Por eso el primero depende de que los códigos estén bien y el segundo de que las geometrías sean válidas y estén en el mismo CRS.

Proyecté ambas capas a EPSG:9377 antes del `overlay` y usé `make_valid=False`, que obliga a resolver la validez de forma visible (sección 6). Con `keep_geom_type=False` pueden aparecer contactos de área cero (puntos o líneas donde solo se tocan los bordes); los registré aparte y conservé solo las piezas poligonales con área mayor que una tolerancia explícita, calculando las hectáreas desde la geometría de cada pieza. La tabla por polígono IGAC compara el área original con la suma intersectada: "parcial" es normal si parte del polígono cae fuera de los municipios disponibles.

**Límites:** si los polígonos IGAC se solapan entre sí, la suma de áreas no es cobertura única, solo una suma de piezas; el diagnóstico compara esa suma con el área de la unión. Cinco polígonos son una muestra: no permiten estimar la capacidad de uso de un municipio completo, y `area_intersectada_ha = 0` solo significa "sin intersección con esta muestra". Tampoco se puede asignar una clase IGAC a una muestra individual de AGROSAVIA, porque esas muestras no tienen coordenadas propias (las de DIVIPOLA son territoriales).

## 9. Publicación y relectura · 35 minutos

Guarda `analisis_integrados.parquet` en OUT con `to_parquet(engine="pyarrow", compression="snappy", index=False)`. Incluye todas las filas, columnas originales, tipos definidos en el contrato, indicadores y procedencia. Usa nombres inequívocos para las coordenadas territoriales. Conserva `codigo_departamento` y `codigo_municipio` como texto; los códigos no resueltos permanecen ausentes.

Vuelve a leer mediante `pd.read_parquet(OUT + "/analisis_integrados.parquet", engine="pyarrow")`. Compara contenido ordenado por `id_fila`, tipos y cantidad de filas. Exporta a CSV controles, equivalencias, resumen y registros pendientes, sin índice. Escribe el contrato en una celda Markdown.

**Control:** reinicia el kernel y reejecuta; la salida debe conservar el mismo contenido. Las escrituras reemplazan archivos concretos de OUT. No modifiques los CSV originales.

Exporta `mapa_municipal` e `intersecciones` como capas separadas con llamadas explícitas:

```python
mapa_municipal.to_file(
    OUT + "/resultados_espaciales.gpkg", layer="municipios",
    driver="GPKG", engine="pyogrio", index=False
)
intersecciones.to_file(
    OUT + "/resultados_espaciales.gpkg", layer="intersecciones",
    driver="GPKG", engine="pyogrio", index=False
)
```
 Convierte explícitamente a texto las columnas categóricas, como el indicador del merge, antes de exportar si el controlador lo requiere. En cada reejecución crea un GeoPackage nuevo o reemplaza solo las capas indicadas; no uses append.

Relee cada capa con `gpd.read_file(OUT + "/resultados_espaciales.gpkg", layer="municipios", engine="pyogrio")` y la llamada equivalente para intersecciones. Verifica filas, claves, CRS, geometría activa, valores y áreas con tolerancia. Los tipos restaurados por GeoPackage pueden variar: documenta las diferencias. El Parquet tabular que usa PD05 conserva todas las observaciones y no lleva la geometría municipal repetida en cada análisis.


In [10]:
# ---------- 9.1 Tabla final: todas las filas, columnas originales, tipos del contrato ----------
analisis_final = analisis_int.copy()
analisis_final["fuente_analisis"] = "agrosavia.csv"
analisis_final["fuente_territorial"] = np.where(analisis_final["codigo_municipio"].notna(), "divipola.csv (corte 2024)", None)
analisis_final = analisis_final.rename(columns={"clave_dep": "clave_departamento_norm", "clave_mun": "clave_municipio_norm"})
columnas_finales = (["id_fila"] + suelo.columns.tolist() +
    ["fecha_dt", "anio", "ph", "ph_valido", "codigo_departamento", "codigo_municipio",
     "nombre_departamento_territorial", "nombre_municipio_territorial", "longitud_territorial", "latitud_territorial",
     "clave_departamento_norm", "clave_municipio_norm", "estado_union", "estado_union_previo", "union_via",
     "merge_indicador", "fuente_analisis", "fuente_territorial"])
assert len(columnas_finales) == len(set(columnas_finales)), "Hay nombres de columna repetidos en la tabla final"
analisis_final = analisis_final[columnas_finales]
columnas_texto = [c for c in columnas_finales if c not in ("id_fila", "fecha_dt", "anio", "ph", "ph_valido",
                                                           "longitud_territorial", "latitud_territorial")]
analisis_final[columnas_texto] = analisis_final[columnas_texto].astype("string")      # texto con <NA>
analisis_final["id_fila"] = analisis_final["id_fila"].astype("int64")
print(analisis_final.shape, "| filas = entrada:", len(analisis_final) == len(suelo))
print(analisis_final.dtypes.astype(str).to_dict())

# ---------- 9.2 Parquet y relectura ----------
RUTA_PQ = OUT + "/analisis_integrados.parquet"
analisis_final.to_parquet(RUTA_PQ, engine="pyarrow", compression="snappy", index=False)
leido = pd.read_parquet(RUTA_PQ, engine="pyarrow")
a = analisis_final.sort_values("id_fila").reset_index(drop=True)
b = leido.sort_values("id_fila").reset_index(drop=True)
print("Filas:", len(a), len(b), "| tipos iguales:", a.dtypes.astype(str).equals(b.dtypes.astype(str)))
pd.testing.assert_frame_equal(a, b)
print("Relectura Parquet: contenido y tipos idénticos.")

# ---------- 9.3 Controles de CRS/geometrías y conciliación tabla-mapa ----------
control_geometrias = pd.DataFrame([
    {"capa": "DANE municipios", "filas": len(municipios), "crs": str(municipios.crs), "ausentes": int(aus.sum()),
     "vacias": int(vac.sum()), "invalidas_antes": int(invalidas.sum()), "reparadas": int(cambiadas.sum()),
     "apartadas_no_poligonales": len(municipios_apartados), "aptas": len(municipios_validos)},
    {"capa": "IGAC capacidad", "filas": len(capacidad), "crs": str(capacidad.crs), "ausentes": int(aus_c.sum()),
     "vacias": int(vac_c.sum()), "invalidas_antes": int(inv_c.sum()), "reparadas": int(inv_c.sum()),
     "apartadas_no_poligonales": len(cap_presentes) - len(capacidad_valida), "aptas": len(capacidad_valida)},
])
conciliacion_tabla_mapa = pd.DataFrame([{
    "anio": anio_elegido,
    "analisis_resumen_anual": int(resumen_anual["total_analisis"].sum()),
    "analisis_en_mapa": int(mapa_municipal["total_analisis"].sum()),
    "analisis_sin_poligono": int(resumen_anual.loc[resumen_anual["codigo_municipio"].isin(codigos_sin_geom), "total_analisis"].sum()),
    "poligonos_sin_resumen": len(sin_resumen), "poligonos_aptos": len(mapa_municipal)}])
display(control_geometrias); display(conciliacion_tabla_mapa)

# ---------- 9.4 CSV de controles, equivalencias, resumen y pendientes ----------
controles = pd.concat([
    pd.DataFrame(list(cantidades_originales.items()), columns=["control", "valor"]),
    control_conversion.reset_index(names="campo").melt(id_vars="campo", var_name="control", value_name="valor").assign(control=lambda d: d["campo"] + "." + d["control"]).drop(columns="campo"),
    pd.DataFrame(list(conciliacion.items()), columns=["control", "valor"]),
    pd.DataFrame({"control": ["cobertura_enlace_pct_antes", "cobertura_enlace_pct_despues"],
                  "valor": [round(100 * analisis_int["estado_union_previo"].isin(["coincide"]).mean(), 2),
                            round(100 * analisis_int["codigo_municipio"].notna().mean(), 2)]}),
], ignore_index=True)
pendientes = analisis_final.loc[~analisis_final["estado_union"].isin(["coincide", "equivalencia_verificada"]),
                                ["id_fila", S_DEP, S_MUN, "clave_departamento_norm", "clave_municipio_norm", "estado_union"]]
controles.to_csv(OUT + "/controles.csv", index=False, encoding="utf-8-sig")
equivalencias.to_csv(OUT + "/equivalencias.csv", index=False, encoding="utf-8-sig")
resumen_mun.to_csv(OUT + "/resumen_municipal.csv", index=False, encoding="utf-8-sig")
pendientes.to_csv(OUT + "/registros_pendientes.csv", index=False, encoding="utf-8-sig")
control_geometrias.to_csv(OUT + "/control_crs_geometrias.csv", index=False, encoding="utf-8-sig")
conciliacion_tabla_mapa.to_csv(OUT + "/conciliacion_tabla_mapa.csv", index=False, encoding="utf-8-sig")
print("Pendientes:", len(pendientes), "| CSV exportados en", OUT)

# ---------- 9.5 GeoPackage: dos capas, sin append ----------
GPKG = OUT + "/resultados_espaciales.gpkg"
if os.path.exists(GPKG):
    os.remove(GPKG)                       # se crea nuevo en cada ejecución
mapa_municipal["merge_resumen"] = mapa_municipal["merge_resumen"].astype(str)
mapa_municipal.to_file(GPKG, layer="municipios", driver="GPKG", engine="pyogrio", index=False)
intersecciones.to_file(GPKG, layer="intersecciones", driver="GPKG", engine="pyogrio", index=False)

# ---------- 9.6 Relectura y verificación de las capas ----------
mun_rd = gpd.read_file(GPKG, layer="municipios", engine="pyogrio")
int_rd = gpd.read_file(GPKG, layer="intersecciones", engine="pyogrio")
print("Filas municipios:", len(mapa_municipal), len(mun_rd), "| intersecciones:", len(intersecciones), len(int_rd))
print("CRS:", mapa_municipal.crs == mun_rd.crs, "| geometría activa:", mun_rd.geometry.name, int_rd.geometry.name)
print("Claves únicas y completas:", mun_rd["codigo_municipio"].is_unique, set(mun_rd["codigo_municipio"]) == set(mapa_municipal["codigo_municipio"]))

m0 = mapa_municipal.set_index("codigo_municipio").sort_index()
m1 = mun_rd.set_index("codigo_municipio").sort_index()
np.testing.assert_allclose(m0["area_ha"], m1["area_ha"], rtol=1e-9)
np.testing.assert_allclose(m0["area_intersectada_ha"], m1["area_intersectada_ha"], rtol=1e-9)
np.testing.assert_allclose(m0["total_analisis"].fillna(-1), m1["total_analisis"].fillna(-1))
# GeoPackage puede promover Polygon a MultiPolygon en la capa: se compara igualdad topológica, no el tipo
assert m0.geometry.geom_equals(m1.geometry).all()
print("Tipos geométricos antes/después:", mapa_municipal.geom_type.value_counts().to_dict(), mun_rd.geom_type.value_counts().to_dict())
area_rd = int_rd.geometry.area / 10000   # int_rd está en EPSG:9377
np.testing.assert_allclose(area_rd.sum(), intersecciones["area_ha"].sum(), rtol=1e-6)
print("Áreas y geometrías coinciden dentro de tolerancia.")

dif_tipos = pd.DataFrame({"antes": mapa_municipal.drop(columns="geometry").dtypes.astype(str),
                          "despues": mun_rd.drop(columns="geometry").dtypes.astype(str)})
display(dif_tipos[dif_tipos["antes"] != dif_tipos["despues"]])

NameError: name 'analisis_int' is not defined

**Interpretación:**
Guardé `analisis_integrados.parquet` con `to_parquet(engine="pyarrow", compression="snappy", index=False)`, con todas las filas, las columnas originales sin tocar, las columnas tipadas del contrato, los indicadores de unión y la procedencia (`fuente_analisis`, `fuente_territorial`, `union_via`). Las columnas de texto usan el tipo `string` de pandas para que las ausencias sigan siendo `<NA>` tras la relectura; los códigos permanecen como texto y los no resueltos quedan ausentes. Al releer con `pd.read_parquet` comparé el contenido ordenado por `id_fila`, los tipos y el número de filas con `pd.testing.assert_frame_equal`.

Por separado exporté a CSV (sin índice) los controles, las equivalencias, el resumen municipal, los registros pendientes, el control de CRS/geometrías y la conciliación tabla–mapa. Las capas espaciales van al GeoPackage como `municipios` e `intersecciones`, con llamadas explícitas a `to_file`; borro el archivo antes de escribir para que cada ejecución genere un GeoPackage nuevo y no se acumulen capas con `append`. Convertí el indicador del merge a texto antes de exportar.

**Diferencias al releer:** en el GeoPackage los enteros nulables y los tipos de texto pueden restaurarse distinto (por ejemplo, `Int64` vuelve como entero de NumPy o como `float` si hay ausentes), por eso comparé valores con tolerancia en lugar de exigir tipos idénticos; la tabla final de diferencias de tipos las documenta. Las coordenadas de las geometrías se conservan; las áreas se verificaron con `assert_allclose`.

**Límites:** el Parquet tabular de PD05 conserva todas las observaciones y no repite la geometría municipal en cada análisis; la unión con los polígonos se hace por `codigo_municipio`. Una relectura exitosa demuestra que el archivo conserva lo escrito, no que el contenido sea correcto. No modifiqué los CSV originales.

## Entrega

`soluciones/PD04_limpieza_integracion_territorial.ipynb`, con contrato, código visible, controles y tres conclusiones. Los resultados regenerables se guardan en `kit/salidas/soluciones/PD04/`. PD05 utiliza `analisis_integrados.parquet`; no elimines registros para facilitar esa actividad.

Incluye también los dos mapas PNG, el GeoPackage con municipios e intersecciones, el control de CRS y geometrías y la conciliación entre resumen tabular y mapa. PD05 reutiliza el Parquet de análisis y la capa municipal. Apoyo: [guía GeoPandas](../docs/manual-geopandas.md), [uniones de atributos y espaciales](https://geopandas.org/en/stable/docs/user_guide/mergingdata.html).
